In [ ]:
!pip install faiss-cpu #install FAISS

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('train.csv')

print("Creating nowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 72.1 MB/s eta 0:00:00
Creating nowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [ ]:
#Q1
zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
row_150 = train.iloc[150]

prompt_150 = str(row_150["prompt"])

labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]

ans_150 = str(row_150[row_150["answer"]])

print("Ground Truth Answer:")
print(ans_150)

Ground Truth Answer:
The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."


In [ ]:
result = zs(
    prompt_150,
    candidate_labels=labels_150,
    multi_label=False
)

# Find the probability assigned to the ground-truth answer
score = result["scores"][result["labels"].index(ans_150)]

print("Ground Truth Option:",ans_150)

print("\nProbability Score:",round(score, 3))

Ground Truth Option: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."

Probability Score: 0.384


In [ ]:
#2
# Embed the prompt
query_embedding = model.encode([prompt_150]).astype("float32")

# Search the FAISS index
k = 10
distances, indices = index.search(query_embedding, k)

print("Top-10 Retrieved Indices:")
print(indices[0])

Top-10 Retrieved Indices:
[ 663 1701 1269 1532  576  847 1693 1906  168  150]


In [ ]:
true_index = 150

retrieved = indices[0]

if true_index in retrieved:
    rank = list(retrieved).index(true_index) + 1
    print("True document found at rank:", rank)
else:
    print("True document is NOT in the Top-10 retrieved documents.")

True document found at rank: 10


In [ ]:
#3
from sentence_transformers import CrossEncoder
import numpy as np

# Load the Cross-Encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Top-10 indices retrieved by FAISS
retrieved_indices = indices[0]

# Get the corresponding documents
docs_10 = [kb[i] for i in retrieved_indices]

# Create (query, document) pairs
pairs = [[prompt_150, doc] for doc in docs_10]

# Score each pair
ce_scores = cross_encoder.predict(pairs)

# Sort by score (highest first)
sorted_order = np.argsort(ce_scores)[::-1]

print("Cross-Encoder Ranking:")
for rank, idx in enumerate(sorted_order, start=1):
    print(
        f"Rank {rank}: KB Index = {retrieved_indices[idx]}, Score = {ce_scores[idx]:.4f}"
    )

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder Ranking:
Rank 1: KB Index = 150, Score = 4.7585
Rank 2: KB Index = 1906, Score = 4.7526
Rank 3: KB Index = 847, Score = 4.7526
Rank 4: KB Index = 1693, Score = 4.7526
Rank 5: KB Index = 1269, Score = 4.7375
Rank 6: KB Index = 1532, Score = 4.7375
Rank 7: KB Index = 168, Score = 4.7072
Rank 8: KB Index = 576, Score = 4.6870
Rank 9: KB Index = 1701, Score = 4.6602
Rank 10: KB Index = 663, Score = 4.6602


In [ ]:
true_index = 150

for rank, idx in enumerate(sorted_order, start=1):
    if retrieved_indices[idx] == true_index:
        print("True document rank:", rank)
        break

True document rank: 1


In [ ]:
#4
# Row 42
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

# Embed the prompt
query_embedding = model.encode([prompt_42]).astype("float32")

# Retrieve top-5 documents
k = 5
distances, indices = index.search(query_embedding, k)

retrieved_indices = indices[0]
retrieved_docs = [kb[i] for i in retrieved_indices]

print("Retrieved Indices:", retrieved_indices)

Retrieved Indices: [241 439 456 506 605]


In [ ]:
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Concatenate documents with a single space
concatenated_docs = " ".join(retrieved_docs)

# Create the final input string
text = f"Context: {concatenated_docs} Question: {prompt_42}"

# Tokenize without truncation
tokens = tokenizer(text, truncation=False)

print("Total Tokens:", len(tokens["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Total Tokens: 216


In [ ]:
#5
# Retrieve the true document (KB index 150)
true_document = kb[150]

# Create the RAG input
rag_prompt = f"Context: {true_document} Question: {prompt_150}"

print(rag_prompt)

Context: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos." Question: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.


In [ ]:
result_rag = zs(
    rag_prompt,
    candidate_labels=labels_150,
    multi_label=False
)

print(result_rag)

# Probability assigned to the ground-truth option
rag_score = result_rag["scores"][result_rag["labels"].index(ans_150)]

print("Ground Truth:", ans_150)
print("Probability:", round(rag_score, 3))

{'sequence': 'Context: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos." Question: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.', 'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on 

In [ ]:
#6
# Force an incorrect context from KB index 999
wrong_document = kb[999]

adversarial_prompt = f"Context: {wrong_document} Question: {prompt_150}"

print(adversarial_prompt)

Context: A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pass from one side to the other, causing a reduce in temperature in one part and an boost in temperature in the other, contrary to the second law of thermodynamics. Question: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.


In [ ]:
result_adv = zs(
    adversarial_prompt,
    candidate_labels=labels_150,
    multi_label=False
)

print(result_adv)

# Probability assigned to the ground-truth correct option
adv_score = result_adv["scores"][result_adv["labels"].index(ans_150)]

print("Ground Truth:", ans_150)
print("Probability:", round(adv_score, 3))

{'sequence': 'Context: A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pass from one side to the other, causing a reduce in temperature in one part and an boost in temperature in the other, contrary to the second law of thermodynamics. Question: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.', 'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz i

In [ ]:
#7
hits = 0
total = 100

for i in range(total):
    row = train.iloc[i]

    prompt = str(row["prompt"])

    # Ground-truth document/text
    correct_doc = str(row[row["answer"]])

    # Embed the prompt
    query_embedding = model.encode([prompt]).astype("float32")

    # Retrieve top-5 documents
    distances, indices = index.search(query_embedding, 5)

    retrieved_docs = [kb[idx] for idx in indices[0]]

    # Check if the exact correct document appears in the retrieved documents
    if correct_doc in retrieved_docs:
        hits += 1

hit_rate = (hits / total) * 100

print("Hits:", hits)
print("Hit Rate:", round(hit_rate, 1))

Hits: 73
Hit Rate: 73.0


In [ ]:
#8
import numpy as np

# Load Cross Encoder (if not already loaded)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def map_at_3(actual, predicted):
    """
    actual: correct letter (e.g. 'C')
    predicted: list of top-3 predicted letters
    """
    for i, p in enumerate(predicted):
        if p == actual:
            return 1.0 / (i + 1)
    return 0.0

scores = []

for i in range(20):
    row = train.iloc[i]

    prompt = str(row["prompt"])
    actual = row["answer"]

    labels = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    # ---------------------------
    # Step 1: Retrieve Top-5
    # ---------------------------
    query_embedding = model.encode([prompt]).astype("float32")
    distances, indices = index.search(query_embedding, 5)

    retrieved_indices = indices[0]
    docs = [kb[idx] for idx in retrieved_indices]

    # ---------------------------
    # Step 2: Cross-Encoder Rerank
    # ---------------------------
    pairs = [[prompt, doc] for doc in docs]
    ce_scores = cross_encoder.predict(pairs)

    best_doc = docs[np.argmax(ce_scores)]

    # ---------------------------
    # Step 3: Build RAG Prompt
    # ---------------------------
    rag_prompt = f"Context: {best_doc} Question: {prompt}"

    # ---------------------------
    # Step 4: Zero-shot Prediction
    # ---------------------------
    result = zs(
        rag_prompt,
        candidate_labels=labels,
        multi_label=False
    )

    # Map label text back to option letters
    text_to_letter = {
        str(row["A"]): "A",
        str(row["B"]): "B",
        str(row["C"]): "C",
        str(row["D"]): "D",
        str(row["E"]): "E",
    }

    ranked_letters = [text_to_letter[label] for label in result["labels"]]

    top3 = ranked_letters[:3]

    score = map_at_3(actual, top3)
    scores.append(score)

print("Average MAP@3:", round(np.mean(scores), 3))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Average MAP@3: 0.975
